# Movie Recommendation System - Popularity Baseline
## Group 5 - AIL303m - Machine Learning | FPT University

**Syllabus Mapping:**
- 18 Dimensionality Reduction & Matrix Factorization (Introduction)

**Objective:** Establish a Popularity-Based Recommendation baseline and Global Mean baseline metrics.
This serves as the **original project / starting point** before improvement with NMF (see `02_NMF_Collaborative_Filtering.ipynb`).

**EDA:** See `EDA.ipynb` for full exploratory data analysis.

## 1. Configurations


In [30]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from datetime import datetime

sns.set_style('darkgrid')
params = {
    'legend.fontsize': 'medium',
    'figure.figsize': (12, 8),
    'figure.dpi': 100,
    'axes.labelsize': 'medium',
    'axes.titlesize': 'large',
    'xtick.labelsize': 'medium',
    'ytick.labelsize': 'medium'
}
plt.rcParams.update(params)

print('Loaded!')

Loaded!


## 2. Data Loading
The MovieLens ml-latest-small dataset contains:
- **100,836 ratings** across **9,742 movies** by **610 users**
- Rating scale: 0.5 to 5.0 (half-star increments)
- Timeframe: March 1996 - September 2018

In [31]:
DATA_PATH = '../data/ml-latest-small/'

rating_df = pd.read_csv(DATA_PATH + 'ratings.csv')
movie_df = pd.read_csv(DATA_PATH + 'movies.csv')

print(f'Ratings shape: {rating_df.shape}')
print(f'Movies shape: {movie_df.shape}')

Ratings shape: (100836, 4)
Movies shape: (9742, 3)


In [ ]:
# Merge ratings with movie titles for popularity analysis
merged_df = pd.merge(rating_df, movie_df, on='movieId')
print(f'Merged dataset shape: {merged_df.shape}')
merged_df.head()

## 5. Popularity-Based Recommendation

### Popularity Baseline:

**Formula:** `weighted_score = (avg_rating * num_votes) / (num_votes + threshold)`

This approach recommends the most popular movies to ALL users (no personalization).
It serves as a **baseline** and **cold-start fallback**.

In [42]:
movie_stats = merged_df.groupby(['movieId', 'title']).agg(
    avg_rating=('rating', 'mean'),
    num_ratings=('rating', 'count')
).reset_index()

threshold = movie_stats['num_ratings'].quantile(0.9)
print(f'Threshold (90th percentile): {threshold:.0f} ratings')

movie_stats['weighted_score'] = (
    (movie_stats['avg_rating'] * movie_stats['num_ratings']) /
    (movie_stats['num_ratings'] + threshold)
)

top10_popular = movie_stats[movie_stats['num_ratings'] >= threshold].sort_values(
    'weighted_score', ascending=False
).head(10)

print('\nTop-10 Popularity-Based Recommendations (for new/cold-start users):')

print('=' * 95)

for i, (_, row) in enumerate(top10_popular.iterrows(), 1):
    print(f'  {i:2d}. {row["title"]:<50s} '
          f'Avg: {row["avg_rating"]:.2f}  '
          f'Count: {int(row["num_ratings"])}  '
          f'Score: {row["weighted_score"]:.3f}')

Threshold (90th percentile): 27 ratings

Top-10 Popularity-Based Recommendations (for new/cold-start users):
   1. Shawshank Redemption, The (1994)                   Avg: 4.43  Count: 317  Score: 4.081
   2. Pulp Fiction (1994)                                Avg: 4.20  Count: 307  Score: 3.858
   3. Forrest Gump (1994)                                Avg: 4.16  Count: 329  Score: 3.848
   4. Matrix, The (1999)                                 Avg: 4.19  Count: 278  Score: 3.821
   5. Star Wars: Episode IV - A New Hope (1977)          Avg: 4.23  Count: 251  Score: 3.820
   6. Fight Club (1999)                                  Avg: 4.27  Count: 218  Score: 3.802
   7. Silence of the Lambs, The (1991)                   Avg: 4.16  Count: 279  Score: 3.794
   8. Schindler's List (1993)                            Avg: 4.22  Count: 220  Score: 3.763
   9. Godfather, The (1972)                              Avg: 4.29  Count: 192  Score: 3.760
  10. Usual Suspects, The (1995)                      

In [43]:
from sklearn.metrics import mean_squared_error

global_mean = rating_df['rating'].mean()

baseline_rmse = np.sqrt(mean_squared_error(
    rating_df['rating'],
    np.full(len(rating_df), global_mean)
))

baseline_mae = np.mean(np.abs(rating_df['rating'] - global_mean))

print(f'Baseline (Global Mean) Performance:')
print(f'  Global Mean Rating: {global_mean:.4f}')
print(f'  Baseline RMSE: {baseline_rmse:.4f}')
print(f'  Baseline MAE: {baseline_mae:.4f}')

Baseline (Global Mean) Performance:
  Global Mean Rating: 3.5016
  Baseline RMSE: 1.0425
  Baseline MAE: 0.8271


## Summary — Baseline Results

### Popularity-Based Recommendation:
- Recommends the **same popular movies** to ALL users (no personalization)
- Uses weighted score formula: `(avg_rating × num_votes) / (num_votes + threshold)`
- Good as a **cold-start fallback** but cannot capture individual preferences

### Global Mean Baseline Metrics:
- **RMSE** and **MAE** computed using Global Mean as prediction for all ratings
- These serve as the **lower bound** — any improvement model must beat these numbers

### Limitation:
- ❌ **No personalization** — every user gets the same recommendations
- ❌ Cannot capture individual taste patterns

**Next Step:** See `02_NMF_Collaborative_Filtering.ipynb` for NMF-based personalized recommendations that improve upon this baseline.